<a href="https://colab.research.google.com/github/johnsoncapitalventuresllc-source/hand-drawn-card-game/blob/main/JCV_Market_Research_Bot_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JCV Market Research & Intelligence Bot v2

Paste your regenerated API keys into **Google Colab → Secrets** using these names:

- `NEWSAPI_KEY`
- `ALPHA_VANTAGE_KEY`
- `FRED_API_KEY`
- `POLYGON_API_KEY`

Enable notebook access for each secret. Run the cells from top to bottom. The final cell runs one briefing immediately.


In [1]:
!pip install pandas requests pytz --quiet

In [6]:
# ============================================================
# CELL 2 — IMPORTS, SECURE CONFIGURATION, AND JCV UNIVERSE
# ============================================================

import os
import re
import csv
import json
import time
from urllib.parse import urlparse
from datetime import datetime, timedelta

import pytz
import requests
import pandas as pd

try:
    from google.colab import userdata
except ImportError:
    userdata = None

EASTERN = pytz.timezone("America/New_York")
COINGECKO_BASE = "https://api.coingecko.com/api/v3"
ALPHA_BASE = "https://www.alphavantage.co/query"
FRED_BASE = "https://api.stlouisfed.org/fred"
POLYGON_DIVIDENDS_URL = "https://api.polygon.io/v3/reference/dividends"
NEWSAPI_EVERYTHING_URL = "https://newsapi.org/v2/everything"

def load_secret(name):
    """Load from Colab Secrets first, then environment variables."""
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    return value or os.getenv(name)

NEWSAPI_KEY = load_secret("NEWSAPI_KEY")
ALPHA_VANTAGE_KEY = load_secret("ALPHA_VANTAGE_KEY")
FRED_API_KEY = load_secret("FRED_API_KEY")
POLYGON_API_KEY = load_secret("POLYGON_API_KEY")

API_KEYS = {
    "NewsAPI": NEWSAPI_KEY,
    "Alpha Vantage": ALPHA_VANTAGE_KEY,
    "FRED": FRED_API_KEY,
    "Polygon/Massive": POLYGON_API_KEY,
}

JCV_DIVIDEND_ETFS = ["SCHD", "VYM", "JEPI"]
JCV_MONTHLY_INCOME = ["O"]
JCV_DIVIDEND_BLUE_CHIPS = [
    "PG", "JNJ", "KO", "PEP", "ABBV", "CVX", "XOM", "JPM",
    "HD", "LOW", "WMT", "COST", "MCD", "MRK", "IBM"
]
JCV_WATCHLIST = sorted(set(
    JCV_DIVIDEND_ETFS + JCV_MONTHLY_INCOME + JCV_DIVIDEND_BLUE_CHIPS
))

MARKET_INDICATORS = {
    "S&P 500 ETF": "SPY",
    "Nasdaq-100 ETF": "QQQ",
    "Russell 2000 ETF": "IWM",
    "Gold ETF": "GLD",
    "Regional Banks ETF": "KRE",
    "Financial Sector ETF": "XLF",
}

BANK_SYMBOLS = ["KRE", "XLF", "JPM", "BAC", "WFC"]

STRATEGIC_TOPICS = {
    "AI & Automation": '(artificial intelligence OR enterprise AI OR automation)',
    "Robotics & Drones": '(robotics OR autonomous robot OR commercial drone OR security drone)',
    "Cybersecurity": '(cybersecurity OR ransomware OR zero trust OR identity security)',
    "Fintech & Payments": '(fintech OR digital payments OR payment infrastructure)',
    "Media & Creator Tech": '("creator economy" OR media technology OR content platform)',
    "Consumer Products": '("consumer products" OR CPG OR retail brand)',
}

BLOCKED_DOMAINS = {
    "pypi.org", "medium.com", "addicted2success.com", "winteriscoming.net",
    "fool.com", "benzinga.com"
}
BLOCKED_TERMS = {
    "price prediction", "could explode", "next 100x", "presale", "airdrop",
    "game of thrones", "valorant", "esports", "skin", "championship"
}
TRUSTED_SOURCE_HINTS = {
    "reuters", "associated press", "ap news", "bloomberg", "cnbc",
    "financial times", "wall street journal", "the verge", "techcrunch",
    "coindesk", "the block", "fortune", "business wire", "globe newswire"
}

REQUEST_TIMEOUT = 25
QUOTE_CACHE = {}
RUN_DIAGNOSTICS = []
API_CALL_COUNTS = {}

def now_eastern():
    return datetime.now(EASTERN)

def add_diagnostic(source, section, status, message):
    RUN_DIAGNOSTICS.append({
        "time": now_eastern().strftime("%Y-%m-%d %H:%M:%S"),
        "source": source,
        "section": section,
        "status": status,
        "message": str(message)[:500],
    })

def increment_api_count(source):
    API_CALL_COUNTS[source] = API_CALL_COUNTS.get(source, 0) + 1

def validate_keys():
    missing = [name for name, value in API_KEYS.items() if not value]
    if missing:
        print("Missing Colab Secrets:", ", ".join(missing))
    else:
        print("All four API secrets were loaded.")
    return missing

validate_keys()


All four API secrets were loaded.


[]

In [8]:
# ============================================================
# CELL 3 — SHARED REQUESTS, LOGGING, FRESHNESS, AND NEWS FILTERS
# ============================================================

def safe_get_json(url, params=None, headers=None, source="API", section="General"):
    """Return structured status instead of silently failing."""
    increment_api_count(source)
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=REQUEST_TIMEOUT,
        )

        if response.status_code == 429:
            message = "Rate limited (HTTP 429)."
            add_diagnostic(source, section, "rate_limited", message)
            return {"ok": False, "status": "rate_limited", "data": None, "message": message}

        if response.status_code in (401, 403):
            message = f"Authentication/permission failure (HTTP {response.status_code})."
            add_diagnostic(source, section, "auth_error", message)
            return {"ok": False, "status": "auth_error", "data": None, "message": message}

        response.raise_for_status()
        data = response.json()
        return {"ok": True, "status": "success", "data": data, "message": ""}

    except requests.Timeout:
        message = f"Request timed out after {REQUEST_TIMEOUT} seconds."
    except requests.RequestException as exc:
        message = f"Request failed: {exc}"
    except ValueError as exc:
        message = f"Invalid JSON response: {exc}"
    except Exception as exc:
        message = f"Unexpected error: {exc}"

    add_diagnostic(source, section, "error", message)
    return {"ok": False, "status": "error", "data": None, "message": message}

def log_rows(rows, csv_file):
    """Append normalized records to a CSV file."""
    if not rows:
        return
    frame = pd.DataFrame(rows)
    file_exists = os.path.exists(csv_file)
    frame.to_csv(csv_file, mode="a", header=not file_exists, index=False)

def parse_percent(value):
    if value is None:
        return None
    try:
        return float(str(value).replace("%", "").strip())
    except (TypeError, ValueError):
        return None

def parse_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def days_old(date_string):
    try:
        observed = datetime.strptime(date_string, "%Y-%m-%d").date()
        return (now_eastern().date() - observed).days
    except Exception:
        return None

def freshness_label(date_string, stale_after_days):
    age = days_old(date_string)
    if age is None:
        return "Unknown freshness"
    if age > stale_after_days:
        return f"STALE ({age} days old)"
    return f"Current ({age} days old)"

def domain_from_url(url):
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def normalize_text(*parts):
    return " ".join(str(x or "") for x in parts).lower()

def is_blocked_article(article):
    title = article.get("title") or ""
    description = article.get("description") or ""
    content = normalize_text(title, description)
    domain = domain_from_url(article.get("url") or "")
    if domain in BLOCKED_DOMAINS:
        return True
    return any(term in content for term in BLOCKED_TERMS)

def source_quality_score(article):
    source = normalize_text(article.get("source", {}).get("name"), domain_from_url(article.get("url") or ""))
    if any(hint in source for hint in TRUSTED_SOURCE_HINTS):
        return 2
    if source and source != "[removed]":
        return 1
    return 0

def deduplicate_articles(articles):
    seen = set()
    unique = []
    for article in articles:
        title = re.sub(r"[^a-z0-9 ]", "", (article.get("title") or "").lower())
        key = " ".join(title.split()[:12])
        if not key or key in seen:
            continue
        seen.add(key)
        unique.append(article)
    return unique

def newsapi_search(query, section, page_size=25, from_days=3):
    if not NEWSAPI_KEY:
        message = "NewsAPI key missing."
        add_diagnostic("NewsAPI", section, "missing_key", message)
        return {"status": "error", "items": [], "message": message}

    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": min(page_size, 100),
        "from": (now_eastern() - timedelta(days=from_days)).strftime("%Y-%m-%d"),
        "apiKey": NEWSAPI_KEY,
    }
    result = safe_get_json(
        NEWSAPI_EVERYTHING_URL,
        params=params,
        source="NewsAPI",
        section=section,
    )
    if not result["ok"]:
        return {"status": "error", "items": [], "message": result["message"]}

    data = result["data"] or {}
    if data.get("status") == "error":
        message = data.get("message", "NewsAPI returned an error.")
        add_diagnostic("NewsAPI", section, "error", message)
        return {"status": "error", "items": [], "message": message}

    articles = data.get("articles", [])
    articles = [a for a in articles if not is_blocked_article(a)]
    return {
        "status": "success",
        "items": deduplicate_articles(articles),
        "message": "",
    }


In [9]:
# ============================================================
# CELL 4 — MARKET, MACRO, DIVIDENDS, BANKS, MOVERS
# ============================================================

def alpha_quote(symbol):
    """Cached Alpha Vantage GLOBAL_QUOTE request."""
    if symbol in QUOTE_CACHE:
        return QUOTE_CACHE[symbol]

    if not ALPHA_VANTAGE_KEY:
        add_diagnostic("Alpha Vantage", f"Quote {symbol}", "missing_key", "API key missing.")
        return None

    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": symbol,
        "apikey": ALPHA_VANTAGE_KEY,
    }
    result = safe_get_json(
        ALPHA_BASE,
        params=params,
        source="Alpha Vantage",
        section=f"Quote {symbol}",
    )
    if not result["ok"]:
        return None

    data = result["data"] or {}
    if data.get("Information") or data.get("Note"):
        message = data.get("Information") or data.get("Note")
        add_diagnostic("Alpha Vantage", f"Quote {symbol}", "rate_limited", message)
        return None

    quote = data.get("Global Quote") or {}
    price = parse_float(quote.get("05. price"))
    change_pct = parse_percent(quote.get("10. change percent"))
    latest_day = quote.get("07. latest trading day")

    if price is None:
        add_diagnostic("Alpha Vantage", f"Quote {symbol}", "empty", "Usable quote not returned.")
        return None

    item = {
        "symbol": symbol,
        "price": price,
        "change_percent": change_pct,
        "latest_trading_day": latest_day,
        "source": "Alpha Vantage",
    }
    QUOTE_CACHE[symbol] = item
    return item

def get_bitcoin_market():
    params = {
        "ids": "bitcoin",
        "vs_currencies": "usd",
        "include_24hr_change": "true",
        "include_last_updated_at": "true",
    }
    result = safe_get_json(
        f"{COINGECKO_BASE}/simple/price",
        params=params,
        source="CoinGecko",
        section="Bitcoin Market",
    )
    if not result["ok"]:
        return {"status": "error", "item": None, "message": result["message"]}

    try:
        btc = result["data"]["bitcoin"]
        updated = datetime.fromtimestamp(
            int(btc["last_updated_at"]), tz=pytz.UTC
        ).astimezone(EASTERN)
        return {
            "status": "success",
            "item": {
                "symbol": "BTC",
                "price": float(btc["usd"]),
                "change_percent": float(btc["usd_24h_change"]),
                "updated_at": updated.strftime("%Y-%m-%d %H:%M:%S %Z"),
                "source": "CoinGecko",
            },
            "message": "",
        }
    except Exception as exc:
        message = f"Bitcoin response parsing failed: {exc}"
        add_diagnostic("CoinGecko", "Bitcoin Market", "parse_error", message)
        return {"status": "error", "item": None, "message": message}

def get_market_dashboard():
    items = []
    unavailable = []

    for name, symbol in MARKET_INDICATORS.items():
        quote = alpha_quote(symbol)
        if quote:
            items.append({"name": name, **quote})
        else:
            unavailable.append(symbol)

    btc = get_bitcoin_market()
    if btc["status"] == "success":
        items.append({"name": "Bitcoin", **btc["item"]})
    else:
        unavailable.append("BTC")

    status = "success" if items else "error"
    return {
        "status": status,
        "items": items,
        "unavailable": unavailable,
        "message": "" if items else "No market dashboard data was available.",
    }

FRED_SERIES = {
    "CPIAUCSL": {
        "name": "Consumer Price Index",
        "unit": "index",
        "stale_days": 50,
    },
    "CPILFESL": {
        "name": "Core Consumer Price Index",
        "unit": "index",
        "stale_days": 50,
    },
    "UNRATE": {
        "name": "Unemployment Rate",
        "unit": "%",
        "stale_days": 50,
    },
    "FEDFUNDS": {
        "name": "Federal Funds Rate",
        "unit": "%",
        "stale_days": 50,
    },
    "DGS10": {
        "name": "10-Year Treasury Yield",
        "unit": "%",
        "stale_days": 7,
    },
    "T10Y2Y": {
        "name": "10Y–2Y Treasury Spread",
        "unit": "%",
        "stale_days": 7,
    },
}

def fred_latest_two(series_id, config):
    if not FRED_API_KEY:
        return {"status": "error", "item": None, "message": "FRED key missing."}

    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "sort_order": "desc",
        "limit": 20,
    }
    result = safe_get_json(
        f"{FRED_BASE}/series/observations",
        params=params,
        source="FRED",
        section=config["name"],
    )
    if not result["ok"]:
        return {"status": "error", "item": None, "message": result["message"]}

    observations = (result["data"] or {}).get("observations", [])
    valid = [
        obs for obs in observations
        if obs.get("value") not in (None, "", ".")
        and parse_float(obs.get("value")) is not None
    ]

    if len(valid) < 1:
        message = "No valid FRED observations returned."
        add_diagnostic("FRED", config["name"], "empty", message)
        return {"status": "error", "item": None, "message": message}

    # sort_order=desc means index 0 is the newest valid observation.
    latest = valid[0]
    previous = valid[1] if len(valid) > 1 else None
    latest_value = parse_float(latest["value"])
    previous_value = parse_float(previous["value"]) if previous else None

    if previous_value is None:
        direction = "Unknown"
    elif latest_value > previous_value:
        direction = "Rising"
    elif latest_value < previous_value:
        direction = "Falling"
    else:
        direction = "Unchanged"

    item = {
        "series_id": series_id,
        "name": config["name"],
        "value": latest_value,
        "previous_value": previous_value,
        "date": latest["date"],
        "previous_date": previous.get("date") if previous else None,
        "direction": direction,
        "unit": config["unit"],
        "freshness": freshness_label(latest["date"], config["stale_days"]),
        "age_days": days_old(latest["date"]),
        "source": "FRED",
    }
    return {"status": "success", "item": item, "message": ""}

def get_macro_indicators():
    items, failures = [], []
    for series_id, config in FRED_SERIES.items():
        result = fred_latest_two(series_id, config)
        if result["status"] == "success":
            items.append(result["item"])
        else:
            failures.append({"series_id": series_id, "message": result["message"]})
    return {
        "status": "success" if items else "error",
        "items": items,
        "failures": failures,
        "message": "" if items else "Macro indicators unavailable.",
    }

def get_dividend_calendar(days_forward=30):
    if not POLYGON_API_KEY:
        message = "Polygon/Massive key missing."
        add_diagnostic("Polygon/Massive", "Dividend Calendar", "missing_key", message)
        return {"status": "error", "items": [], "message": message}

    start = now_eastern().date()
    end = start + timedelta(days=days_forward)
    params = {
        "ex_dividend_date.gte": start.isoformat(),
        "ex_dividend_date.lte": end.isoformat(),
        "limit": 1000,
        "sort": "ex_dividend_date",
        "order": "asc",
        "apiKey": POLYGON_API_KEY,
    }
    result = safe_get_json(
        POLYGON_DIVIDENDS_URL,
        params=params,
        source="Polygon/Massive",
        section="Dividend Calendar",
    )
    if not result["ok"]:
        return {"status": "error", "items": [], "message": result["message"]}

    data = result["data"] or {}
    items = []
    for record in data.get("results", []):
        ticker = record.get("ticker")
        if ticker not in JCV_WATCHLIST:
            continue
        items.append({
            "ticker": ticker,
            "cash_amount": record.get("cash_amount"),
            "currency": record.get("currency", "USD"),
            "ex_date": record.get("ex_dividend_date"),
            "pay_date": record.get("pay_date"),
            "declaration_date": record.get("declaration_date"),
            "frequency": record.get("frequency"),
            "dividend_type": record.get("dividend_type"),
            "source": "Polygon/Massive",
        })

    return {
        "status": "success",
        "items": items,
        "message": (
            f"{len(items)} JCV watchlist dividend event(s) found."
            if items else
            "API succeeded; no JCV watchlist dividend events were found."
        ),
    }

def banking_risk_label(items):
    changes = [
        x.get("change_percent") for x in items
        if x.get("change_percent") is not None
    ]
    if not changes:
        return "Unknown", "Insufficient quote data."
    worst = min(changes)
    average = sum(changes) / len(changes)
    if worst <= -7 or average <= -4:
        return "Severe", "Large broad banking-sector decline."
    if worst <= -4 or average <= -2:
        return "Elevated", "Meaningful banking-sector weakness."
    if worst <= -2 or average <= -1:
        return "Moderate", "Some banking-sector pressure."
    return "Low", "No broad banking stress signal from daily price changes."

def get_banking_health():
    items = []
    for symbol in BANK_SYMBOLS:
        quote = alpha_quote(symbol)
        if quote:
            items.append(quote)
    risk, interpretation = banking_risk_label(items)
    return {
        "status": "success" if items else "error",
        "items": items,
        "risk": risk,
        "interpretation": interpretation,
        "message": "" if items else "Banking quotes unavailable.",
    }

def valid_mover_symbol(symbol):
    if not symbol:
        return False
    symbol = symbol.upper().strip()
    if not re.fullmatch(r"[A-Z]{1,5}", symbol):
        return False
    if symbol.endswith(("W", "U", "R")):
        return False
    return True

def verified_ticker_catalyst(ticker):
    search = newsapi_search(f'"{ticker}" AND (stock OR shares)', f"Mover Catalyst {ticker}", page_size=8, from_days=2)
    if search["status"] != "success":
        return None

    ticker_pattern = re.compile(rf"(?<![A-Z]){re.escape(ticker)}(?![A-Z])", re.IGNORECASE)
    for article in search["items"]:
        combined = f"{article.get('title') or ''} {article.get('description') or ''}"
        if not ticker_pattern.search(combined):
            continue
        if source_quality_score(article) < 1:
            continue
        return {
            "headline": article.get("title"),
            "source": article.get("source", {}).get("name"),
            "url": article.get("url"),
            "published_at": article.get("publishedAt"),
        }
    return None

def get_verified_major_movers(max_each=3, minimum_price=5.0):
    if not ALPHA_VANTAGE_KEY:
        return {"status": "error", "gainers": [], "losers": [], "message": "Alpha Vantage key missing."}

    params = {
        "function": "TOP_GAINERS_LOSERS",
        "apikey": ALPHA_VANTAGE_KEY,
    }
    result = safe_get_json(
        ALPHA_BASE,
        params=params,
        source="Alpha Vantage",
        section="Top Movers",
    )
    if not result["ok"]:
        return {"status": "error", "gainers": [], "losers": [], "message": result["message"]}

    data = result["data"] or {}
    if data.get("Information") or data.get("Note"):
        message = data.get("Information") or data.get("Note")
        add_diagnostic("Alpha Vantage", "Top Movers", "rate_limited", message)
        return {"status": "error", "gainers": [], "losers": [], "message": message}

    def process(records):
        accepted = []
        for record in records:
            if len(accepted) >= max_each:
                break
            ticker = (record.get("ticker") or "").upper()
            price = parse_float(record.get("price"))
            change = parse_percent(record.get("change_percentage"))
            if not valid_mover_symbol(ticker):
                continue
            if price is None or price < minimum_price:
                continue

            catalyst = verified_ticker_catalyst(ticker)
            if catalyst is None:
                continue

            accepted.append({
                "ticker": ticker,
                "price": price,
                "change_percent": change,
                "catalyst": catalyst,
            })
        return accepted

    gainers = process(data.get("top_gainers", []))
    losers = process(data.get("top_losers", []))
    return {
        "status": "success",
        "gainers": gainers,
        "losers": losers,
        "message": (
            "Only movers with a minimum price and a ticker-matched recent catalyst are shown."
        ),
    }


In [10]:
# ============================================================
# CELL 5 — BITCOIN NEWS, STRATEGIC NEWS, VC, RISK, ACTION BOARD
# ============================================================

def score_bitcoin_article(article):
    text = normalize_text(article.get("title"), article.get("description"))
    score = source_quality_score(article)

    direct_terms = {
        "bitcoin": 2, "btc": 2, "spot bitcoin etf": 3,
        "bitcoin etf": 3, "coinbase": 2, "sec": 1,
        "cftc": 1, "stablecoin legislation": 2,
        "bitcoin mining": 2, "institutional custody": 2,
        "blackrock": 1, "strategy": 1, "microstrategy": 1,
        "exchange hack": 3, "exchange failure": 3,
    }
    for term, points in direct_terms.items():
        if term in text:
            score += points

    if "bitcoin" not in text and "btc" not in text:
        score -= 3
    return score

def get_bitcoin_intelligence(max_items=5):
    query = (
        '(bitcoin OR BTC OR "spot bitcoin ETF" OR Coinbase OR "bitcoin mining") '
        'AND (regulation OR SEC OR CFTC OR ETF OR institutional OR custody OR hack OR tax OR Federal Reserve)'
    )
    search = newsapi_search(query, "Bitcoin Intelligence", page_size=40, from_days=3)
    if search["status"] != "success":
        return search

    scored = []
    for article in search["items"]:
        score = score_bitcoin_article(article)
        if score < 4:
            continue
        scored.append({
            "title": article.get("title"),
            "description": article.get("description"),
            "source": article.get("source", {}).get("name"),
            "url": article.get("url"),
            "published_at": article.get("publishedAt"),
            "relevance_score": score,
        })

    scored.sort(key=lambda x: (x["relevance_score"], x.get("published_at") or ""), reverse=True)
    return {
        "status": "success",
        "items": scored[:max_items],
        "message": (
            f"{len(scored[:max_items])} high-relevance Bitcoin article(s) selected."
            if scored else
            "API succeeded; no Bitcoin articles passed the relevance threshold."
        ),
    }

def strategic_article_score(article, topic_name):
    text = normalize_text(article.get("title"), article.get("description"))
    score = source_quality_score(article)
    topic_terms = {
        "AI & Automation": ["artificial intelligence", "enterprise ai", "automation"],
        "Robotics & Drones": ["robot", "robotics", "drone", "autonomous"],
        "Cybersecurity": ["cybersecurity", "ransomware", "zero trust", "identity security"],
        "Fintech & Payments": ["fintech", "payment", "financial infrastructure"],
        "Media & Creator Tech": ["creator economy", "media technology", "content platform"],
        "Consumer Products": ["consumer products", "cpg", "retail brand"],
    }
    score += sum(2 for term in topic_terms.get(topic_name, []) if term in text)
    return score

def get_strategic_sector_news(max_per_topic=2):
    results = []
    for topic_name, query in STRATEGIC_TOPICS.items():
        search = newsapi_search(query, f"Strategic News: {topic_name}", page_size=20, from_days=4)
        if search["status"] != "success":
            continue

        candidates = []
        for article in search["items"]:
            score = strategic_article_score(article, topic_name)
            if score < 3:
                continue
            candidates.append({
                "topic": topic_name,
                "title": article.get("title"),
                "source": article.get("source", {}).get("name"),
                "url": article.get("url"),
                "published_at": article.get("publishedAt"),
                "score": score,
            })
        candidates.sort(key=lambda x: (x["score"], x.get("published_at") or ""), reverse=True)
        results.extend(candidates[:max_per_topic])

    return {
        "status": "success",
        "items": results,
        "message": (
            f"{len(results)} strategic-sector article(s) selected."
            if results else
            "No strategic-sector articles passed the relevance threshold."
        ),
    }

FUNDING_TERMS = [
    "series a", "series b", "series c", "seed round",
    "funding round", "venture funding", "raised $", "raises $",
]
JCV_SECTOR_TERMS = [
    "artificial intelligence", "ai ", "robot", "drone", "cybersecurity",
    "fintech", "payment", "creator", "media technology", "consumer",
    "logistics", "automation",
]

def funding_article_score(article):
    text = normalize_text(article.get("title"), article.get("description"))
    score = source_quality_score(article)
    if any(term in text for term in FUNDING_TERMS):
        score += 3
    if re.search(r"\$\s?\d+(\.\d+)?\s?(million|billion|m|b)\b", text):
        score += 2
    if any(term in text for term in JCV_SECTOR_TERMS):
        score += 2
    return score

def get_vc_funding_rounds(max_items=5):
    query = (
        '("Series A" OR "Series B" OR "Series C" OR "seed round" OR "funding round") '
        'AND (AI OR robotics OR drones OR cybersecurity OR fintech OR payments OR automation OR "creator economy")'
    )
    search = newsapi_search(query, "VC Funding", page_size=40, from_days=7)
    if search["status"] != "success":
        return search

    items = []
    for article in search["items"]:
        score = funding_article_score(article)
        if score < 6:
            continue
        items.append({
            "title": article.get("title"),
            "description": article.get("description"),
            "source": article.get("source", {}).get("name"),
            "url": article.get("url"),
            "published_at": article.get("publishedAt"),
            "score": score,
        })

    items.sort(key=lambda x: (x["score"], x.get("published_at") or ""), reverse=True)
    return {
        "status": "success",
        "items": items[:max_items],
        "message": (
            f"{len(items[:max_items])} validated funding article(s) selected."
            if items else
            "No funding articles passed the required transaction and relevance checks."
        ),
    }

def score_market_risk(market, macro, banks):
    score = 3
    reasons = []

    market_by_symbol = {x.get("symbol"): x for x in market.get("items", [])}
    spy_change = market_by_symbol.get("SPY", {}).get("change_percent")
    qqq_change = market_by_symbol.get("QQQ", {}).get("change_percent")
    kre_change = market_by_symbol.get("KRE", {}).get("change_percent")
    btc_change = market_by_symbol.get("BTC", {}).get("change_percent")

    for label, value in [("SPY", spy_change), ("QQQ", qqq_change)]:
        if value is not None and value <= -2:
            score += 1
            reasons.append(f"{label} is down at least 2% today.")
        elif value is not None and value >= 2:
            score -= 0.5

    if kre_change is not None and kre_change <= -3:
        score += 1
        reasons.append("Regional banks are under meaningful daily pressure.")

    if btc_change is not None and btc_change <= -7:
        score += 1
        reasons.append("Bitcoin volatility is elevated.")

    macro_by_id = {x.get("series_id"): x for x in macro.get("items", [])}
    ten_year = macro_by_id.get("DGS10", {})
    spread = macro_by_id.get("T10Y2Y", {})
    unemployment = macro_by_id.get("UNRATE", {})

    if ten_year.get("direction") == "Rising" and (ten_year.get("value") or 0) >= 4.5:
        score += 1
        reasons.append("The 10-year Treasury yield is elevated and rising.")

    if spread.get("value") is not None and spread["value"] < 0:
        score += 1
        reasons.append("The 10Y–2Y yield curve remains inverted.")

    if unemployment.get("direction") == "Rising":
        score += 0.5
        reasons.append("The latest unemployment observation increased.")

    bank_risk = banks.get("risk")
    if bank_risk == "Moderate":
        score += 0.5
    elif bank_risk == "Elevated":
        score += 1.5
    elif bank_risk == "Severe":
        score += 2.5

    score = max(1, min(10, round(score)))
    if score <= 3:
        condition = "Low"
    elif score <= 5:
        condition = "Normal to cautious"
    elif score <= 7:
        condition = "Elevated"
    else:
        condition = "High"

    if not reasons:
        reasons.append("No major stress rule was triggered by available data.")

    return {"score": score, "condition": condition, "reasons": reasons}

def build_action_board(risk, dividends, macro, banks):
    score = risk["score"]

    if score <= 5:
        dividend_action = "Continue standard accumulation; do not accelerate solely from this report."
        btc_action = "Continue standard DCA; no change based on current risk score."
        cash_action = "Maintain planned cash buffer."
    elif score <= 7:
        dividend_action = "Continue cautiously; review large purchases manually."
        btc_action = "Maintain base DCA; avoid increasing size without manual review."
        cash_action = "Favor liquidity and preserve the cash buffer."
    else:
        dividend_action = "Pause any unplanned increase and require executive review."
        btc_action = "Keep only the pre-approved base DCA or pause for manual review."
        cash_action = "Prioritize liquidity."

    stale_macro = [x for x in macro.get("items", []) if "STALE" in x.get("freshness", "")]
    manual_review = []
    if stale_macro:
        manual_review.append(f"{len(stale_macro)} stale macro indicator(s)")
    if banks.get("risk") in ("Elevated", "Severe"):
        manual_review.append(f"banking risk is {banks.get('risk').lower()}")
    if dividends.get("status") == "error":
        manual_review.append("dividend API failed")

    return {
        "Dividend Strategy": dividend_action,
        "Bitcoin Strategy": btc_action,
        "Options Strategy": "No automated options action. This briefing is research-only.",
        "Cash Reserves": cash_action,
        "Manual Review": ", ".join(manual_review) if manual_review else "None required by current rules.",
    }


In [11]:
# ============================================================
# CELL 6 — BRIEFING BUILDER, FILE EXPORT, AND MANUAL RUN
# ============================================================

def money(value):
    if value is None:
        return "N/A"
    return f"${value:,.2f}"

def pct(value):
    if value is None:
        return "N/A"
    return f"{value:+.2f}%"

def number(value):
    if value is None:
        return "N/A"
    return f"{value:,.3f}".rstrip("0").rstrip(".")

def format_article(article):
    return (
        f"{article.get('title', 'Untitled')} | "
        f"Source: {article.get('source') or 'Unknown'} | "
        f"Published: {article.get('published_at') or 'Unknown'} | "
        f"Link: {article.get('url') or 'N/A'}"
    )

def build_briefing():
    global RUN_DIAGNOSTICS, API_CALL_COUNTS, QUOTE_CACHE
    RUN_DIAGNOSTICS = []
    API_CALL_COUNTS = {}
    QUOTE_CACHE = {}

    generated = now_eastern()
    date_stamp = generated.strftime("%Y-%m-%d")
    timestamp = generated.strftime("%Y-%m-%d %H:%M:%S %Z")

    briefing_file = f"jcv_briefing_{date_stamp}.txt"
    intelligence_log = "jcv_market_intelligence_log.csv"
    diagnostics_log = "jcv_api_diagnostics_log.csv"

    print("Collecting JCV briefing data...")

    market = get_market_dashboard()
    macro = get_macro_indicators()
    dividends = get_dividend_calendar()
    banks = get_banking_health()
    movers = get_verified_major_movers()
    bitcoin_news = get_bitcoin_intelligence()
    strategic_news = get_strategic_sector_news()
    vc_rounds = get_vc_funding_rounds()

    risk = score_market_risk(market, macro, banks)
    action_board = build_action_board(risk, dividends, macro, banks)

    lines = []
    add = lines.append

    add("=" * 72)
    add("JCV EXECUTIVE MARKET RESEARCH & INTELLIGENCE BRIEFING")
    add(f"Generated: {timestamp}")
    add("Research-only output. No trades are placed by this notebook.")
    add("=" * 72)
    add("")

    add("1. EXECUTIVE SUMMARY")
    add("-" * 72)
    add(f"Overall Market Risk: {risk['score']}/10 — {risk['condition']}")
    add(f"Banking Risk: {banks.get('risk', 'Unknown')}")
    add(f"Dividend Calendar: {dividends.get('message', 'Unavailable')}")
    add("Primary Risk Drivers:")
    for reason in risk["reasons"]:
        add(f"- {reason}")
    add("")

    add("2. JCV ACTION BOARD")
    add("-" * 72)
    for name, action in action_board.items():
        add(f"{name}: {action}")
    add("")

    add("3. MARKET DASHBOARD")
    add("-" * 72)
    if market["items"]:
        for item in market["items"]:
            update = item.get("latest_trading_day") or item.get("updated_at") or "Unknown"
            add(
                f"{item['name']} ({item['symbol']}): {money(item['price'])} | "
                f"Change: {pct(item.get('change_percent'))} | "
                f"Data time/date: {update} | Source: {item['source']}"
            )
    else:
        add(f"DATA WARNING: {market['message']}")
    if market.get("unavailable"):
        add(f"Unavailable instruments: {', '.join(market['unavailable'])}")
    add("")

    add("4. MACRO INDICATORS")
    add("-" * 72)
    if macro["items"]:
        for item in macro["items"]:
            suffix = item["unit"] if item["unit"] != "index" else ""
            current = f"{number(item['value'])}{suffix}"
            previous = f"{number(item['previous_value'])}{suffix}"
            add(
                f"{item['name']}: {current} | Previous: {previous} | "
                f"Direction: {item['direction']} | Observation: {item['date']} | "
                f"Freshness: {item['freshness']} | Source: FRED"
            )
    else:
        add(f"DATA WARNING: {macro['message']}")
    if macro.get("failures"):
        add(f"Failed macro series: {len(macro['failures'])}")
    add("")

    add("5. DIVIDEND CALENDAR — JCV WATCHLIST, NEXT 30 DAYS")
    add("-" * 72)
    if dividends["status"] == "error":
        add(f"DATA WARNING: {dividends['message']}")
    elif dividends["items"]:
        for item in dividends["items"]:
            add(
                f"{item['ticker']} | Cash amount: {money(item['cash_amount'])} "
                f"{item.get('currency') or ''} | Ex-date: {item.get('ex_date') or 'N/A'} | "
                f"Pay date: {item.get('pay_date') or 'N/A'} | "
                f"Frequency code: {item.get('frequency') or 'N/A'} | Source: Polygon/Massive"
            )
    else:
        add(dividends["message"])
    add("")

    add("6. BANKING SECTOR HEALTH")
    add("-" * 72)
    add(f"Risk: {banks.get('risk', 'Unknown')} | Interpretation: {banks.get('interpretation', '')}")
    if banks["items"]:
        for item in banks["items"]:
            add(
                f"{item['symbol']}: {money(item['price'])} | "
                f"Change: {pct(item.get('change_percent'))} | "
                f"Trading date: {item.get('latest_trading_day') or 'Unknown'}"
            )
    else:
        add(f"DATA WARNING: {banks['message']}")
    add("")

    add("7. VERIFIED MAJOR MARKET MOVERS")
    add("-" * 72)
    add(movers.get("message", ""))
    add("Gainers:")
    if movers.get("gainers"):
        for item in movers["gainers"]:
            catalyst = item["catalyst"]
            add(
                f"{item['ticker']} | Price: {money(item['price'])} | "
                f"Change: {pct(item['change_percent'])} | "
                f"Verified ticker-matched catalyst: {catalyst['headline']} | "
                f"Source: {catalyst['source']} | Link: {catalyst['url']}"
            )
    else:
        add("No gainers passed all filters.")
    add("Losers:")
    if movers.get("losers"):
        for item in movers["losers"]:
            catalyst = item["catalyst"]
            add(
                f"{item['ticker']} | Price: {money(item['price'])} | "
                f"Change: {pct(item['change_percent'])} | "
                f"Verified ticker-matched catalyst: {catalyst['headline']} | "
                f"Source: {catalyst['source']} | Link: {catalyst['url']}"
            )
    else:
        add("No losers passed all filters.")
    add("")

    add("8. BITCOIN INTELLIGENCE")
    add("-" * 72)
    if bitcoin_news["status"] == "error":
        add(f"DATA WARNING: {bitcoin_news['message']}")
    elif bitcoin_news["items"]:
        for article in bitcoin_news["items"]:
            add(f"{format_article(article)} | Relevance score: {article['relevance_score']}")
    else:
        add(bitcoin_news["message"])
    add("")

    add("9. JCV STRATEGIC SECTOR NEWS")
    add("-" * 72)
    if strategic_news["items"]:
        for article in strategic_news["items"]:
            add(f"[{article['topic']}] {format_article(article)}")
    else:
        add(strategic_news["message"])
    add("")

    add("10. VALIDATED VC FUNDING ACTIVITY")
    add("-" * 72)
    if vc_rounds["status"] == "error":
        add(f"DATA WARNING: {vc_rounds['message']}")
    elif vc_rounds["items"]:
        for article in vc_rounds["items"]:
            add(format_article(article))
    else:
        add(vc_rounds["message"])
    add("")

    add("11. UPCOMING ECONOMIC EVENTS")
    add("-" * 72)
    add(
        "Not yet connected. FRED observations are historical indicators, not an "
        "upcoming-release calendar. No upcoming event dates are inferred."
    )
    add("")

    add("12. DATA QUALITY REPORT")
    add("-" * 72)
    add("API calls this run:")
    for source, count in sorted(API_CALL_COUNTS.items()):
        add(f"- {source}: {count}")
    if RUN_DIAGNOSTICS:
        add(f"Warnings/errors recorded: {len(RUN_DIAGNOSTICS)}")
        for diagnostic in RUN_DIAGNOSTICS:
            add(
                f"- [{diagnostic['status'].upper()}] {diagnostic['source']} / "
                f"{diagnostic['section']}: {diagnostic['message']}"
            )
    else:
        add("No API errors or warnings were recorded.")
    add("")

    add("=" * 72)
    add("END OF JCV BRIEFING")
    add("=" * 72)

    briefing = "\n".join(lines)

    with open(briefing_file, "w", encoding="utf-8") as file:
        file.write(briefing)

    market_log_rows = []
    for item in market.get("items", []):
        market_log_rows.append({
            "timestamp_eastern": timestamp,
            "section": "Market Dashboard",
            "name": item.get("name"),
            "symbol": item.get("symbol"),
            "value": item.get("price"),
            "change_percent": item.get("change_percent"),
            "source": item.get("source"),
        })
    for item in macro.get("items", []):
        market_log_rows.append({
            "timestamp_eastern": timestamp,
            "section": "Macro",
            "name": item.get("name"),
            "symbol": item.get("series_id"),
            "value": item.get("value"),
            "change_percent": None,
            "source": item.get("source"),
        })
    log_rows(market_log_rows, intelligence_log)
    log_rows(RUN_DIAGNOSTICS, diagnostics_log)

    print("\n" + briefing)
    print(f"\nSaved briefing: {briefing_file}")
    print(f"Saved intelligence log: {intelligence_log}")
    print(f"Saved diagnostics log: {diagnostics_log}")

    return {
        "briefing": briefing,
        "briefing_file": briefing_file,
        "intelligence_log": intelligence_log,
        "diagnostics_log": diagnostics_log,
        "risk": risk,
        "actions": action_board,
    }

# Run one complete test now:
RESULT = build_briefing()



JCV EXECUTIVE MARKET RESEARCH & INTELLIGENCE BRIEFING
Generated: 2026-07-13 22:51:01 EDT
Research-only output. No trades are placed by this notebook.

1. EXECUTIVE SUMMARY
------------------------------------------------------------------------
Overall Market Risk: 4/10 — Normal to cautious
Banking Risk: Low
Dividend Calendar: 4 JCV watchlist dividend event(s) found.
Primary Risk Drivers:
- The 10-year Treasury yield is elevated and rising.

2. JCV ACTION BOARD
------------------------------------------------------------------------
Dividend Strategy: Continue standard accumulation; do not accelerate solely from this report.
Bitcoin Strategy: Continue standard DCA; no change based on current risk score.
Options Strategy: No automated options action. This briefing is research-only.
Cash Reserves: Maintain planned cash buffer.
Manual Review: 2 stale macro indicator(s)

3. MARKET DASHBOARD
------------------------------------------------------------------------
S&P 500 ETF (SPY): $749.17